# L4b: Breadth-First and Depth-First Search

L4a introduced graph representations and the distinction between vertices, edges, and stored weights. In this lab, we turn one directed edge list into an adjacency list, implement deterministic depth-first and breadth-first traversal, and test how the starting vertex and worklist discipline change the order we observe.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Construct the connectivity used by a traversal:__ Parse a directed edge list, build a graph model, and reduce its weighted edges to a sorted adjacency list. Explain why DFS and BFS use the source and target identifiers but do not use the edge weights.
> * __Implement deterministic DFS and BFS:__ Complete recursive depth-first and queue-based breadth-first traversals in [`src/Compute.jl`](src/Compute.jl). Use a visited set to process every reachable vertex once, and examine outgoing neighbors in ascending identifier order so the result is reproducible.
> * __Test and interpret traversal behavior:__ Verify ordinary, cyclic, and invalid-start cases; distinguish first-visit order from a graph path; and explain why a directed traversal returns only the vertices reachable from its selected start.

Let's get started!
___

## Algorithms

The two traversal algorithms differ in the collection that holds discovered work. That one choice changes the first-visit order even when the graph, starting vertex, visited-set rule, and neighbor ordering are identical.

* __Depth-first search__ follows one branch recursively until it cannot continue, then backtracks to the most recent unfinished vertex. [Read the DFS algorithm notebook](CHEME-5800-L4b-Algorithm-DepthFirstSearch-Fall-2026.ipynb).
* __Breadth-first search__ uses a first-in, first-out queue to process all vertices at one directed distance before advancing to the next layer. [Read the BFS algorithm notebook](CHEME-5800-L4b-Algorithm-BreadthFirstSearch-Fall-2026.ipynb).

The lab has three tasks:

1. Build and validate the adjacency list for the directed example graph.
2. Complete the DFS and BFS implementations in [`src/Compute.jl`](src/Compute.jl).
3. Test the two contracts and interpret their traversal orders.

___

## Setup, Data, and Prerequisites

The setup file activates the pinned course environment, loads the student implementation from [`src/Compute.jl`](src/Compute.jl), and imports the packages used in this lab.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. This meeting-local file uses paths relative to its own location, loads the course package and [the `L4bTraversal` module](src/Compute.jl), and collects the imports in one place.

Run the setup cell before beginning the lab:

In [ ]:
# Load this lab's file-relative environment, student source, and imports.
include(joinpath(@__DIR__, "Include.jl"));

The setup loads [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) for executable checks, [the `DataFrames.jl` package](https://dataframes.juliadata.org/stable/) and [the `PrettyTables.jl` package](https://ronisbr.github.io/PrettyTables.jl/stable/) for readable comparisons, and [the `L4bTraversal` module](src/Compute.jl) from [`src/Compute.jl`](src/Compute.jl). The module is deliberately reloaded whenever this setup cell runs, so edits saved during Task 2 become available without restarting the kernel. Notebook calls remain qualified; for example, [the `L4bTraversal.depth_first_order(...)` function](src/Compute.jl) always refers to the method in the most recently loaded module.

___

## Task 1: Build and validate the directed graph
In this task, we will build and validate the directed graph representation that the DFS and BFS implementations use. The edge list in [`data/SimpleGraph.txt`](data/SimpleGraph.txt) encodes the six-vertex graph shown below. Each record stores a source vertex, a target vertex, and a weight. We will retain the weights in [a `MySimpleDirectedGraphModel` object](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySimpleDirectedGraphModel), then build the simpler adjacency list that the traversal functions accept.

<div>
    <center>
        <img src="figs/Fig-Example-Graph.svg" width="480" alt="Six-vertex directed graph used to compare depth-first and breadth-first traversal"/>
    </center>
</div>

> __What information does a traversal use?__
>
> The graph model stores every directed edge as `(source, target) => weight`. Both traversal algorithms answer a connectivity question: what can be reached, and in what first-visit order? The adjacency list used by both algorithms therefore keeps only each source vertex and its outgoing target vertices. The numerical weights remain available for the shortest-path calculations in L4c, but they do not affect either traversal in this lab.

First define the callback that parses one non-comment record from the edge file:

In [ ]:
"""
    parse_edge_record(record::String, delimiter::Char = ',')

Parse `source,target,weight` text into the tuple expected by
`MyGraphEdgeModels(...)`. Return `nothing` for a malformed record.
"""
function parse_edge_record(record::String, delimiter::Char = ',')
    fields = strip.(split(record, delimiter)); # remove whitespace around the three fields
    length(fields) == 3 || return nothing

    source = parse(Int, fields[1]);       # directed-edge source identifier
    target = parse(Int, fields[2]);       # directed-edge target identifier
    weight = parse(Float64, fields[3]);   # retained for later weighted-graph algorithms
    return (source, target, weight)
end

Load the edge records, construct the course graph model, and pass the directed source-target pairs to [the completed `adjacency_from_edges(...)` helper](src/Compute.jl). That helper creates a key for every vertex, removes duplicate edges, and sorts each outgoing-neighbor vector.

In [ ]:
# Load the weighted edge records from the file shown in the schematic.
edge_file = joinpath(CHEME5800_L4B_DATA, "SimpleGraph.txt");
edge_models = MyGraphEdgeModels(edge_file, parse_edge_record; delim = ',', comment = '#');

# Retain the complete weighted representation in the course graph model.
directed_graph = build(MySimpleDirectedGraphModel, edge_models);

# Reduce each edge to connectivity only, then normalize the adjacency list.
edge_pairs = [(edge.source, edge.target) for edge in values(edge_models)];
adjacency = L4bTraversal.adjacency_from_edges(edge_pairs);

Display one row per vertex. An empty outgoing-neighbor entry means that a traversal can reach the vertex but cannot continue forward from it.

In [ ]:
vertices = sort!(collect(keys(adjacency)));
adjacency_table = DataFrame(
    vertex = vertices,
    outgoing_neighbors = [isempty(adjacency[v]) ? "∅" : join(adjacency[v], ", ") for v in vertices],
);
pretty_table(adjacency_table)

The structural check below connects the file, graph model, and adjacency list. Passing it establishes that the traversal functions will receive the graph shown in the figure, not merely an object that happened to construct without an error.

In [ ]:
@testset "L4b directed-graph representation" begin
    # Check the two representations against the known six-vertex example.
    @test length(directed_graph.nodes) == 6
    @test length(directed_graph.edges) == 7
    @test vertices == collect(1:6)

    # Check every outgoing-neighbor vector used by DFS and BFS.
    @test adjacency == Dict(
        1 => [2, 3],
        2 => [3, 4],
        3 => [5],
        4 => [6],
        5 => [4],
        6 => Int64[],
    )
end;

Starting at vertex 1 can reach every vertex. Starting at vertex 3 can only follow `3 → 5 → 4 → 6`; edge direction prevents it from returning to vertices 1 or 2. Task 3 will turn that visual prediction into an executable reachability check.

___

## Task 2: Implement deterministic DFS and BFS

In this task, we will complete deterministic recursive DFS and queue-based BFS implementations, then check their first traversal orders on the example graph.

The [`depth_first_order(...)`](src/Compute.jl) and [`breadth_first_order(...)`](src/Compute.jl) functions live in [the `L4bTraversal` module](src/Compute.jl) in [`src/Compute.jl`](src/Compute.jl). Their worklist mechanisms differ, but they share one public contract.

> __Traversal-function contract__
>
> __Inputs__
>
> * `adjacency::AbstractDict`: maps every vertex identifier to its outgoing neighbors. Neither function may mutate this dictionary or its neighbor collections.
> * `start::Integer`: the vertex at which traversal begins. A Boolean is not a valid identifier even though `Bool <: Integer` in Julia.
>
> __Output__
>
> * `Vector{Int64}`: every vertex reachable from `start`, recorded exactly once in first-visit order. Outgoing neighbors are considered in ascending identifier order, making the result deterministic.
>
> __Errors__
>
> * [`ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError): `start` is a Boolean or does not appear as a key in `adjacency`.

Use the [DFS algorithm notebook](CHEME-5800-L4b-Algorithm-DepthFirstSearch-Fall-2026.ipynb) and [BFS algorithm notebook](CHEME-5800-L4b-Algorithm-BreadthFirstSearch-Fall-2026.ipynb) while completing the six TODOs in [`src/Compute.jl`](src/Compute.jl):

1. Validate the DFS start and allocate its visited set and result vector.
2. Implement the recursive first-visit and ordered-neighbor logic.
3. Start the recursion and return the DFS order.
4. Validate and enqueue the BFS start, marking it visited immediately.
5. Process the FIFO queue with a head index and record dequeued vertices.
6. Discover, mark, and enqueue each previously unseen ordered neighbor.

Save [`src/Compute.jl`](src/Compute.jl), then rerun the setup cell. Until all six TODOs are complete, the calls below raise the direct implementation errors supplied by the scaffold.

In [ ]:
# Compute first-visit orders from the common starting vertex.
dfs_order = L4bTraversal.depth_first_order(adjacency, 1);
bfs_order = L4bTraversal.breadth_first_order(adjacency, 1);
(depth_first = dfs_order, breadth_first = bfs_order)

Both results should contain all six vertices. The depth-first traversal should return `[1, 2, 3, 5, 4, 6]`, following the first available branch before backtracking. The breadth-first traversal should return `[1, 2, 3, 4, 5, 6]`, processing the vertices discovered one directed edge from vertex 1 before the more distant layers.

___

## Task 3: Test and compare the traversals

In this task, we will test the complete DFS and BFS contracts and compare how the two worklist disciplines order the same reachable vertices.

A traversal order is a record of when vertices were first visited; it is not generally a path through the graph. For example, vertices 3 and 4 are consecutive in the BFS order even though the graph has no edge `3 → 4`. The table below therefore compares the vertex occupying each visit position without labeling consecutive rows as edges or paths.

In [ ]:
# Compare first-visit positions, not nonexistent edges between successive rows.
comparison_table = DataFrame(
    visit_position = collect(eachindex(dfs_order)),
    dfs_vertex = dfs_order,
    bfs_vertex = bfs_order,
    same_vertex = dfs_order .== bfs_order,
);
pretty_table(comparison_table)

Now start both traversals at vertex 3. The reachable subgraph is the single directed chain `3 → 5 → 4 → 6`, so DFS and BFS should return the same order even though their worklists operate differently.

In [ ]:
# Compare the reachable set and first-visit order from an interior vertex.
dfs_from_three = L4bTraversal.depth_first_order(adjacency, 3);
bfs_from_three = L4bTraversal.breadth_first_order(adjacency, 3);
(depth_first = dfs_from_three, breadth_first = bfs_from_three)

The final test set records the complete traversal contract. The additional cyclic graph establishes that the visited set guarantees termination, while a snapshot taken before either cyclic traversal establishes that neither function changes its input.

In [ ]:
# Build a cyclic case with a duplicate edge to exercise discovery logic.
cyclic_adjacency = L4bTraversal.adjacency_from_edges([
    (1, 2), (1, 2), (2, 3), (3, 1), (2, 4),
]);
cyclic_snapshot = deepcopy(cyclic_adjacency); # preserve the input before either traversal runs

@testset "deterministic DFS and BFS contracts" begin
    # Verify the known first-visit orders for the lab graph.
    @test dfs_order == [1, 2, 3, 5, 4, 6]
    @test bfs_order == [1, 2, 3, 4, 5, 6]
    @test dfs_from_three == [3, 5, 4, 6]
    @test bfs_from_three == [3, 5, 4, 6]

    # Verify termination and first-visit behavior in a cyclic graph.
    @test L4bTraversal.depth_first_order(cyclic_adjacency, 1) == [1, 2, 3, 4]
    @test L4bTraversal.breadth_first_order(cyclic_adjacency, 1) == [1, 2, 3, 4]

    # Verify non-mutation and every documented invalid-start category.
    @test cyclic_adjacency == cyclic_snapshot
    @test_throws ArgumentError L4bTraversal.depth_first_order(adjacency, 99)
    @test_throws ArgumentError L4bTraversal.breadth_first_order(adjacency, true)
end;

The tests establish correctness for the specified contract. The interpretation is just as important: the two searches can visit the same reachable set in different orders, while a graph with only one forward branch can make their orders identical. Neighbor ordering is also part of the result; changing the ascending-order rule can change either first-visit sequence without changing the graph itself.

___

## Summary

In this lab, we converted a weighted directed graph into the connectivity needed for traversal, implemented recursive DFS and queue-based BFS, and tested their behavior on ordinary, directed-reachability, cyclic, and invalid-start cases.

> __Key Takeaways:__
>
> * __The worklist determines first-visit order:__ Depth-first search uses the active recursion stack to continue along one branch before backtracking, whereas breadth-first search uses a FIFO queue to process vertices in directed-distance layers. The algorithms can therefore return different orders while visiting exactly the same reachable set.
> * __Discovery state makes graph traversal terminate:__ A visited set ensures each reachable vertex is processed once even when the graph contains cycles or several incoming edges. In BFS, marking a vertex when it is enqueued prevents two frontier vertices from adding duplicate queue entries.
> * __A traversal order is not a path:__ Consecutive vertices in a first-visit sequence need not share an edge. The sequence also depends on the starting vertex, edge directions, and the chosen neighbor-ordering rule, so each of those assumptions belongs in a reproducible traversal contract.

L4c adds edge weights back to the problem: instead of asking only which vertices are reachable and when they are first visited, shortest-path algorithms ask which reachable route has the smallest accumulated cost.

___